# Классификация мусора по фото

Курсовая работа по курсу Computer Vision.

**Задача:** классификация фотографий отходов на 6 классов — картон, стекло, металл, бумага, пластик, прочее.

**Модель:** MobileNetV2 с предобученными весами ImageNet ([статья](https://arxiv.org/abs/1801.04381)).

**Данные:** [Garbage Classification на Kaggle](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification).

## 1. Импорты и настройки

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASSES_RU = {
    'cardboard': 'картон',
    'glass': 'стекло',
    'metal': 'металл',
    'paper': 'бумага',
    'plastic': 'пластик',
    'trash': 'прочее',
}

## 2. Загрузка и разведка данных

Архив `garbage_classification.zip` с Kaggle нужно заранее положить в корень Google Drive.

В архиве датасет лежит дважды: в папках `Garbage classification` и `garbage classification` (отличаются только регистром, файлы одинаковые). Поэтому распаковываем только одну копию.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/garbage_classification.zip'

!unzip -q -n "{ZIP_PATH}" "Garbage classification/*" -d /content/raw

DATA_DIR = '/content/raw/Garbage classification/Garbage classification'
print(sorted(os.listdir(DATA_DIR)))

### Количество изображений по классам

In [ ]:
counts = [len(os.listdir(os.path.join(DATA_DIR, cls))) for cls in CLASSES]

df_counts = pd.DataFrame({
    'класс': [CLASSES_RU[cls] for cls in CLASSES],
    'количество': counts,
})
df_counts['доля, %'] = (df_counts['количество'] / df_counts['количество'].sum() * 100).round(1)

print('Всего изображений:', df_counts['количество'].sum())
df_counts

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(df_counts['класс'], df_counts['количество'], color='steelblue')
for i, value in enumerate(df_counts['количество']):
    plt.text(i, value + 5, str(value), ha='center')
plt.title('Количество изображений по классам')
plt.ylabel('Количество')
plt.tight_layout()
plt.show()

### Примеры изображений

In [ ]:
n_examples = 4
fig, axes = plt.subplots(len(CLASSES), n_examples, figsize=(n_examples * 3, len(CLASSES) * 2.4))

for row, cls in enumerate(CLASSES):
    files = random.sample(os.listdir(os.path.join(DATA_DIR, cls)), n_examples)
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(DATA_DIR, cls, fname))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
    axes[row, 0].set_title(CLASSES_RU[cls], loc='left')

plt.tight_layout()
plt.show()

### Размеры и цветовой режим изображений

In [ ]:
rows = []
for cls in CLASSES:
    for fname in os.listdir(os.path.join(DATA_DIR, cls)):
        with Image.open(os.path.join(DATA_DIR, cls, fname)) as img:
            rows.append({'ширина': img.width, 'высота': img.height, 'режим': img.mode})

pd.DataFrame(rows).value_counts()

### Выводы по данным

- В датасете 2527 изображений, 6 классов.
- Классы несбалансированы: больше всего бумаги (594, 23.5%), меньше всего класса «прочее» (137, 5.4%). Это нужно учесть: делить данные со стратификацией и смотреть не только на accuracy, но и на метрики по классам.
- Все изображения одного размера 512×384 в RGB. В кадре один объект, в основном на однотонном светлом фоне, часть фото сделана крупным планом.
- Некоторые классы визуально похожи: например, прозрачные бутылки из стекла и пластика. Скорее всего, модель будет их путать.
- Данных мало (около 400 фото на класс), поэтому обучать большую сеть с нуля будет тяжело. Это аргумент в пользу transfer learning с предобученной на ImageNet моделью.

## 3. Препроцессинг и разделение на train/val/test

## 4. Бейзлайн: простая CNN

## 5. MobileNetV2 с transfer learning

## 6. Fine-tuning

## 7. Оценка результатов

## 8. Демо

## 9. Выводы